In [ ]:
!pip install yfinance pandas numpy scikit-learn xgboost shap matplotlib -q

In [ ]:
import os, numpy as np, pandas as pd, yfinance as yf, matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, roc_auc_score
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings("ignore")

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)


In [ ]:
# download SPX daily prices
path = "data/raw/spx_daily.csv"
if os.path.exists(path):
    spx = pd.read_csv(path, index_col="Date", parse_dates=True)
else:
    spx = yf.download("^GSPC", start="2005-01-01", end="2025-12-31", auto_adjust=True)
    spx.to_csv(path)

close = spx["Close"].squeeze()
print(f"SPX: {len(close)} trading days")


In [ ]:
# compute realized volatility at 1-day, 5-day, and 22-day horizons
log_ret = np.log(close / close.shift(1))
squared = log_ret ** 2

rv = pd.DataFrame(index=close.index)
for h in [1, 5, 22]:
    rv[f"RV_{h}d"] = np.sqrt(squared.rolling(h).sum() * (252 / h))
rv = rv.dropna()

print(f"RV features: {rv.shape}")
rv.head()


In [ ]:
# build target: 22-day forward realized vol + spike label
fwd_rv = []
sq = squared.values
for i in range(len(sq)):
    end = i + 1 + 22
    if end > len(sq):
        fwd_rv.append(np.nan)
    else:
        fwd_rv.append(np.sqrt(np.sum(sq[i+1:end]) * (252 / 22)))

targets = pd.DataFrame({"fwd_rv_22d": fwd_rv}, index=close.index).dropna()
threshold = np.percentile(targets["fwd_rv_22d"], 90)
targets["spike_label"] = (targets["fwd_rv_22d"] > threshold).astype(int)

print(f"Targets: {targets.shape}")
print(f"Spike threshold (p90): {threshold:.4f}")
print(f"Spikes: {targets['spike_label'].sum()} / {len(targets)}")


In [ ]:
# merge features and targets
data = rv.join(targets, how="inner").dropna()
har_cols = ["RV_1d", "RV_5d", "RV_22d"]

# train/test split: everything before 2020 for training
train = data[data.index < "2020-01-01"]
test = data[data.index >= "2020-01-01"]

print(f"Train: {len(train)} days ({train.index.min().date()} to {train.index.max().date()})")
print(f"Test:  {len(test)} days ({test.index.min().date()} to {test.index.max().date()})")


In [ ]:
# HAR-RV linear regression
har = LinearRegression()
har.fit(train[har_cols], train["fwd_rv_22d"])
har_pred = har.predict(test[har_cols])

har_mse = mean_squared_error(test["fwd_rv_22d"], har_pred)
har_mae = mean_absolute_error(test["fwd_rv_22d"], har_pred)
har_auc = roc_auc_score(test["spike_label"], har_pred)

print(f"Coefficients: RV_1d={har.coef_[0]:.4f}, RV_5d={har.coef_[1]:.4f}, RV_22d={har.coef_[2]:.4f}")
print(f"MSE: {har_mse:.6f}")
print(f"MAE: {har_mae:.6f}")
print(f"AUC: {har_auc:.4f}")


In [ ]:
# predictions vs actual
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test.index, test["fwd_rv_22d"], linewidth=0.8, color="black", alpha=0.6, label="Actual")
ax.plot(test.index, har_pred, linewidth=0.8, color="tab:red", alpha=0.8, label="HAR-RV Predicted")
ax.fill_between(test.index, test["fwd_rv_22d"], har_pred, alpha=0.15, color="red")
ax.set_title("HAR-RV Baseline: Predicted vs Actual (2020+)")
ax.set_ylabel("22-Day Forward RV")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# RV time series plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(rv.index, rv["RV_22d"], linewidth=0.7)
axes[0].set_title("22-Day Realized Volatility")
axes[0].set_ylabel("RV (annualized)")

axes[1].plot(targets.index, targets["fwd_rv_22d"], linewidth=0.7, color="tab:orange")
axes[1].axhline(threshold, color="red", linestyle="--", label=f"Spike threshold ({threshold:.2f})")
axes[1].set_title("22-Day Forward RV (target)")
axes[1].set_ylabel("Forward RV")
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
# save for use in other notebooks
rv.to_csv("data/processed/har_rv_features.csv")
targets.to_csv("data/processed/targets.csv")
print("Saved har_rv_features.csv and targets.csv")
